# A small guarded HR workflow

Use this folder and the shared root kernel. We combine only three learned checks. Inputs, fixture response templates, and separate expectations are AI-authored synthetic teaching data. Expectations are drafts, not human-reviewed judgments.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/inputs.json").read_text())
context = json.loads(Path("docs/dataset/policies.json").read_text())
expectations = json.loads(Path("docs/goldends/draft_expectations.json").read_text())
pprint([(item["id"], item["kind"]) for item in fixtures])

## Read helpers before importing

`tools/checks.py` contains topic keywords, email masking, and citation-ID checks. `tools/workflow.py` fixes their order around a local responder. `models/demo.py` records its actual input and copies a labelled fixture template; it does not generate model output. Durations use real perf_counter measurements and are noisy.

In [ ]:
from tools.workflow import answer_request
from models.demo import fixture_response
print("Local response fixtures only: zero model/API calls.")

## Guards off on all fixtures

Every input reaches the local fixture responder. Inspect actual received inputs and exposed outputs, including the synthetic email. These raw fields are deliberate teaching inspection data.

In [ ]:
off_inputs = []
guards_off = {item["id"]: answer_request(item, context, fixture_response, off_inputs, guarded=False) for item in fixtures}
for name, result in guards_off.items():
    pprint({"id": name, **result})
pprint({"actual_responder_inputs": off_inputs})

## Guards on, same fixtures

Topic rejection skips the responder; email masking runs before it and before display; invalid citations withhold an already-produced fixture answer. Inspect per-check reasons, stages, and actual inputs.

In [ ]:
on_inputs = []
guards_on = {item["id"]: answer_request(item, context, fixture_response, on_inputs, guarded=True) for item in fixtures}
for name, result in guards_on.items():
    pprint({"id": name, **result})
pprint({"actual_responder_inputs": on_inputs})

## Compare outcomes and observed overhead

Count accidental blocks/missed violations against AI-draft decisions. Count the email exposure separately because expected masking is not an expected block. Local timings are actual measurements, not predictions of API latency or cost.

In [ ]:
for label, results, actual_inputs in [("off", guards_off, off_inputs), ("on", guards_on, on_inputs)]:
    accidental = sum(expectations["expected_decisions"][name] == "allow" and result["decision"] == "block" for name, result in results.items())
    missed = sum(expectations["expected_decisions"][name] == "block" and result["decision"] == "allow" for name, result in results.items())
    email_leak = "@" in results["email"]["exposed"] or "@" in results["email"]["responder_input"]
    pprint({"guards": label, "accidental_blocks_vs_AI_draft": accidental, "missed_violations_vs_AI_draft": missed, "email_exposure_failure": email_leak,
            "measured_total_ms": sum(result["duration_ms"] for result in results.values()), "actual_local_calls": len(actual_inputs), "model_api_calls": 0, "guard_model_calls": 0})
assert "@" not in guards_on["email"]["responder_input"] and "@" not in guards_on["email"]["exposed"]

## Change only include_vacation and replay

Repair the accidental vacation block. The unrelated poker claim still passes keyword scope and a valid citation ID: no factual support check ran. Explain that remaining miss and the skipped/local calls; no fixed feedback form is required.

In [ ]:
candidate_inputs = []
candidate = {item["id"]: answer_request(item, context, fixture_response, candidate_inputs, guarded=True, include_vacation=True) for item in fixtures}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "guard_on": guards_on[name]["decision"], "candidate": candidate[name]["decision"], "reason": candidate[name]["reason"], "exposed": candidate[name]["exposed"]})
assert guards_on["borderline"]["decision"] == "block" and candidate["borderline"]["decision"] == "allow"
assert candidate["semantic_bypass"]["decision"] == "allow"
pprint({"candidate_actual_local_calls": len(candidate_inputs), "candidate_measured_total_ms": sum(result["duration_ms"] for result in candidate.values()),
        "candidate_accidental_blocks_vs_AI_draft": sum(expectations["expected_decisions"][name] == "allow" and result["decision"] == "block" for name, result in candidate.items()),
        "candidate_missed_violations_vs_AI_draft": sum(expectations["expected_decisions"][name] == "block" and result["decision"] == "allow" for name, result in candidate.items())})